# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Load the same anonymized starter file used in ML-02.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AsserGharib1/flyrank-internshipML"
REPO_DIR = "flyrank-internshipML"

def at_repo_root():
    return os.path.isdir("data/raw")

if IN_COLAB:
    if not at_repo_root():
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Walk up until we find the repo root. Stop if the directory stops changing,
    # which is what happens at a drive root on Windows and at / on Linux.
    previous = None
    while not at_repo_root() and os.getcwd() != previous:
        previous = os.getcwd()
        os.chdir("..")

assert at_repo_root(), "Could not find data/raw. Open this from inside the repo."

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Rows:", df.shape[0], " Columns:", df.shape[1], " Clients:", df.client_id.nunique())

Working dir: /content/flyrank-internshipML
Rows: 30000  Columns: 44  Clients: 32


## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

**Task type:** ranking. A model or score assigns each eligible page a value, then pages are sorted into a review queue. The reviewer acts on the highest-ranked pages first.

## 2. Target or proxy

My provisional target is **falls behind its own client**. A page is positive when its later-window impression change is at least 20 percentage points worse than the median change for pages from the same client.

I do not use `trend_direction` as the target for the capstone because it is already a rule derived from `trend_pct`. In the starter file the earlier and later 30-day columns only sketch the target. In the warehouse I will build separate feature and outcome windows.

The 20-point cutoff is provisional. A client-relative label also hides site-wide decline, so client-level context remains a limitation.

In [2]:
MIN_PRIOR_IMPRESSIONS = 100
GAP_CUTOFF = -20

mv = df[df["impressions_prev_30d"] >= MIN_PRIOR_IMPRESSIONS].copy()
mv["change_pct"] = (
    (mv["impressions_last_30d"] - mv["impressions_prev_30d"])
    / mv["impressions_prev_30d"] * 100
)
mv["gap_vs_client"] = (
    mv["change_pct"] - mv.groupby("client_id")["change_pct"].transform("median")
)
mv["target_falls_behind"] = (mv["gap_vs_client"] <= GAP_CUTOFF).astype(int)

valid = df["impressions_prev_30d"] > 0
raw_change = (
    (df.loc[valid, "impressions_last_30d"] - df.loc[valid, "impressions_prev_30d"])
    / df.loc[valid, "impressions_prev_30d"] * 100
)
trend_rule_matches = df.loc[valid, "trend_direction"].eq("down").equals(raw_change.lt(-20))

print(f"Eligible rows: {len(mv):,}")
print(f"Target positive rate: {mv.target_falls_behind.mean():.3f}")
print(f"'down' exactly reproduces the -20% rule: {trend_rule_matches}")

Eligible rows: 18,010
Target positive rate: 0.251
'down' exactly reproduces the -20% rule: True


## 3. Success metric

**Primary metric: Precision@50.** It answers the operational question directly: of the first 50 pages a reviewer sees, how many later meet the target?

A good result must beat the transparent rule baseline on the **same held-out clients and the same rows**. The base rate is printed beside the calibration rules so Precision@50 has context.

In [3]:
base_rate = mv["target_falls_behind"].mean()
rules = {
    "fewest prior clicks": mv.nsmallest(50, "clicks_prev_30d"),
    "fewest prior sessions": mv.nsmallest(50, "sessions_prev_30d"),
    "lowest prior impressions": mv.nsmallest(50, "impressions_prev_30d"),
}

print(f"Starter target base rate: {base_rate:.3f}")
for name, selected in rules.items():
    print(f"{name:<26} Precision@50 = {selected.target_falls_behind.mean():.3f}")
print("These are starter-data calibration checks, not the final validation result.")

Starter target base rate: 0.251
fewest prior clicks        Precision@50 = 0.300
fewest prior sessions      Precision@50 = 0.360
lowest prior impressions   Precision@50 = 0.340
These are starter-data calibration checks, not the final validation result.


## 4. The unit of analysis, as a real dataframe

**One row = one content item at one decision point.** IDs are used only for grouping and validation, never as model features.

In the starter data I keep the earlier 30-day measurements as clearly pre-outcome signals. Later-window values, `trend_pct`, and `trend_direction` are excluded from features. The fixed 90-day fields overlap the teaching outcome window, so I do not treat them as safe capstone features.

In [4]:
view_cols = [
    "content_id", "client_id",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "target_falls_behind",
]
unit_df = mv[view_cols].copy()

print(f"Rows: {len(unit_df):,}")
print(f"Distinct content items: {unit_df.content_id.nunique():,}")
print(f"Rows per content item: {len(unit_df) / unit_df.content_id.nunique():.2f}")
print(unit_df.head(5).to_string(index=False))

Rows: 18,010
Distinct content items: 18,010
Rows per content item: 1.00
          content_id         client_id  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  target_falls_behind
content_304f48230142 client_f369cb89fc                   987               13                  9                    0
content_a1fb4e703a9e client_4e07408562                  5915                1                  2                    1
content_9aa793d4d895 client_7f2253d7e2                  6089                3                  3                    0
content_331d6c4de07b client_19581e27de                  4206               17                 26                    0
content_d99b7a2d90ca client_3fdba35f04                  6452                2                  9                    0


## 5. Why ML beats a fixed rule here

I do not assume it does. A model is worth keeping only if combining safe signals improves Precision@50 over a readable rule on held-out clients. If the model does not beat that baseline, the rule is the better result because it is easier to explain and operate.

## Self-check

- [x] Task type, target/proxy, and one success metric are explicit
- [x] The unit of analysis is shown as a dataframe
- [x] IDs and outcome-derived columns are excluded from model features
- [x] ML is treated as something to test against a rule, not as the goal
- [ ] Final corrected notebook committed and repo URL submitted on the ML-03 card